# Buổi 26 — Lab

Chạy từng ô từ trên xuống. Mỗi bước ứng với một mục trong tài liệu (mục 5). Bạn sửa `conformal.py`; các ô tự dùng bản mới.

In [ ]:
# sửa tệp .py trong code/ thì các ô sau tự dùng bản mới, không cần khởi động lại
%load_ext autoreload
%autoreload 2

## Bước 1 — Dữ liệu và mô hình điểm

In [ ]:
%matplotlib inline
import warnings

import conformal as cf
import matplotlib.pyplot as plt
import numpy as np

warnings.simplefilter("ignore")
df = cf.doc_pm25()
P = cf.du_bao(df)                                  # LightGBM học năm 1; dự báo năm hiệu chỉnh + hai năm kiểm
k = (P["giai_doan"] == "kiem").to_numpy()
y = P.loc[k, "y"].to_numpy()
print("giờ hiệu chỉnh:", int((~k).sum()), "| giờ kiểm:", int(k.sum()))
print("MAE LightGBM:", round(np.abs(y - P.loc[k, "yhat"]).mean(), 2), "| naive (giờ trước):", round(np.abs(y - P.loc[k, "lag1"]).mean(), 2))

## Bước 2 — Split conformal (mục 4.1–4.2)

In [ ]:
ks = cf.split_conformal(P)
print({a: round(b, 3) for a, b in cf.tom_tat(y, ks.loc[k, "lo"], ks.loc[k, "hi"]).items()})
trong = (y >= ks.loc[k, "lo"].to_numpy()) & (y <= ks.loc[k, "hi"].to_numpy())
print("coverage theo quý:", {q: round(float(trong[P.loc[k, "ds"].dt.quarter.to_numpy() == q].mean()), 3) for q in (1, 2, 3, 4)})

## Bước 3 — CQR (mục 4.3)

In [ ]:
kc = cf.cqr(P)
print({a: round(b, 3) for a, b in cf.tom_tat(y, kc.loc[k, "lo"], kc.loc[k, "hi"]).items()})
lag1 = P.loc[k, "lag1"].to_numpy()
for ten, K in (("split", ks), ("CQR", kc)):
    trong = (y >= K.loc[k, "lo"].to_numpy()) & (y <= K.loc[k, "hi"].to_numpy())
    print(ten, "| giờ trước ≤ 35:", round(float(trong[lag1 <= 35].mean()), 3), "| giờ trước > 150:", round(float(trong[lag1 > 150].mean()), 3))

## Bước 4 — EnbPI (mục 4.4)

Học 20 LightGBM trên mẫu bootstrap, khoảng 10 giây.

In [ ]:
ke = cf.enbpi(df)
ye = P.set_index("ds")["y"].reindex(ke["ds"]).to_numpy()
print({a: round(b, 3) for a, b in cf.tom_tat(ye, ke["lo"], ke["hi"]).items()})

## Bước 5 — ACI và khoảng triển khai (mục 4.5)

Sửa `aci` và `PHUONG_PHAP` rồi chạy lại ô này.

In [ ]:
ka = cf.aci(P)
print("ACI", {a: round(b, 3) for a, b in cf.tom_tat(y, ka.loc[k, "lo"], ka.loc[k, "hi"]).items()})
kt = cf.khoang_trien_khai(P)
print("triển khai (", cf.PHUONG_PHAP, ")", {a: round(b, 3) for a, b in cf.tom_tat(y, kt.loc[k, "lo"], kt.loc[k, "hi"]).items()})
fig, ax = plt.subplots(figsize=(11, 3.2))
for ten, K in (("split", ks.loc[k]), ("CQR", kc.loc[k]), ("ACI", ka.loc[k])):
    ax.plot(P.loc[k, "ds"], cf.coverage_truot(y, K["lo"].to_numpy(), K["hi"].to_numpy()), lw=0.9, label=ten)
ax.plot(ke["ds"], cf.coverage_truot(ye, ke["lo"].to_numpy(), ke["hi"].to_numpy()), lw=0.8, label="EnbPI")
ax.axhspan(0.85, 0.95, color="gray", alpha=0.15)
ax.set_ylabel("coverage 30 ngày")
ax.legend(ncol=4)
plt.show()

## Bước 6 — MAPIE so với tự viết (mục 4.6)

Khoảng 1 phút.

In [ ]:
km = cf.mapie_aci(df)
cung = (P["ds"] >= cf.HIEU_CHINH_DEN) & (P["ds"] < "2015-09-01")
print("MAPIE   ", {a: round(b, 3) for a, b in cf.tom_tat(km["y"], km["lo"], km["hi"]).items()})
print("tự viết ", {a: round(b, 3) for a, b in cf.tom_tat(P.loc[cung, "y"], ka.loc[cung, "lo"], ka.loc[cung, "hi"]).items()})

## Bước 7 — Kiểm tra

Trong terminal, thư mục `lab/`: `python lab.py check` — xanh 7/7 là xong.